In [1]:
import pandas as pd
import numpy as np

# Daten einlesen

In [ ]:
df = pd.read_csv("../data/processed/swissvotes_processed.csv")

## Kongruenz ohne Mobilisierung (Framing nicht mehr Stimmberechtigte sondern in Bezug auf

In [6]:
df_with_positions = df.copy()
neue_spalten = {}

for col in parteien:
    scores = []

    for i, row in df_with_positions.iterrows():
        position = row[col]
        ja_proz = row['volkja-proz']

        # Keine klare Empfehlung oder keine Daten
        if position not in [1.0, 2.0] or pd.isna(ja_proz):
            scores.append(np.nan)

        # Partei empfiehlt JA → je mehr Ja-Stimmen, desto positiver
        elif position == 1.0:
            scores.append((ja_proz - 50) / 100)

        # Partei empfiehlt NEIN → je mehr Nein-Stimmen, desto positiver
        elif position == 2.0:
            scores.append((50 - ja_proz) / 100)

    neue_spalten[f'zustimmung_{col}'] = scores

df_with_positions = pd.concat(
    [df_with_positions, pd.DataFrame(neue_spalten, index=df_with_positions.index)],
    axis=1
)
print(df_with_positions['zustimmung_br-pos'].dropna().unique())

KeyError: 'zustimmung_br-pos'

In [68]:
df_with_positions.to_csv('../data/processed/df_with_positions.csv', index=False)

### Datenset für Heatmap

In [ ]:
kanton_cols = [c for c in df.columns if c.endswith('-japroz')]

scores = {}
zeile = df.iloc[70]
for k in kanton_cols:
    ja_proz = pd.to_numeric(zeile[k], errors="coerce")

    if position not in (1.0, 2.0) or pd.isna(ja_proz):
        scores[k] = np.nan
    elif position == 1.0:
        scores[k] = (ja_proz - 50) / 100
    else:  # position == 2.0
        scores[k] = (50 - ja_proz) / 100

scores

# Eine Zeile, Spalten = Kantone (oder Kurzlabels)
br_kantone = pd.DataFrame([scores], index=["br-pos"])
# Optional Spalten umbenennen: zh-japroz -> ZH
br_kantone.columns = [c.replace("-japroz", "").upper() for c in br_kantone.columns]
br_kantone

,ZH,BE,LU,UR,SZ,OW,NW,GL,ZG,FR,...,SG,GR,AG,TG,TI,VD,VS,NE,GE,JU
br-pos,0.1351,0.1202,0.2944,0.0932,0.0966,0.1299,0.2391,0.2087,0.0259,-0.0966,...,0.0198,0.074,0.074,-0.0493,0.0404,-0.2361,-0.1551,-0.2078,-0.1087,NaN
